# 20 kV MV ring — transient analysis

Time-domain analysis of the same 20 kV medium-voltage ring as in the
companion notebook **MV ring (stationary)**: 20 substations linked by
0.5 km sections of NA2XS(F)2Y 1x300 RM/25 cable, `R_g = 5 Ohm` per
substation, 1 kA fed at `b01`, fault at `b10`.

`groundinsight` has two transient solvers, and they model different things:

* the **FFT solver** evaluates the frequency-domain impedance formulas per
  frequency bin. It does **not** evaluate the inductive coupling between the
  phase conductor and the shields, and warns when a network has some
  (`TransientModelWarning`, code `TRANSIENT_MODEL`);
* the **state-space solver** integrates the lumped R/L/C fields in time. It
  includes the coupling (`R_mutual_formula`, `M_mutual_formula`) for current
  sources, with the phase current divided between the two directions of the
  ring as in the stationary solve, and it lumps the branch shunt capacitance
  onto the adjacent buses (pi-section).

The notebook checks both against the stationary solve in steady state, shows
the effect of the coupling on a fault current with a DC offset, and switches a
Thévenin source onto the ring. [Transient simulation](../../transient/)
derives both solvers.

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np

import groundinsight as gi
from groundinsight.simulation import waveforms

## 1. Cable and bus parameters

The per-length values of the stationary notebook.

In [ ]:
R_PER_KM = 0.25       # ohm / km, shield-earth loop
L_PER_KM = 1.9e-3     # H / km
C_PER_KM = 300e-9     # F / km
R_MUT_PER_KM = 0.05   # ohm / km, Carson earth-return resistance
M_PER_KM = 1.85e-3    # H / km, phase-shield mutual inductance

SECTION_LENGTH_KM = 0.5
N_BUSES = 20
R_BUS = 5.0
# Every bus accumulates C/2 from each of its two cable sections (pi-section).
C_BUS_LUMPED = C_PER_KM * SECTION_LENGTH_KM

## 2. Build the network — with and without the coupling

`ring(mutual=...)` builds the ring once with the phase-shield coupling and once
without it, for comparison. Both carry the frequency-domain formulas (for the
FFT solver and the stationary solve) and the lumped fields (for the
state-space solver).

In [ ]:
def ring(mutual=True, source="current"):
    bus_type = gi.BusType(
        name="MV_substation", system_type="Substation", voltage_level=20.0,
        impedance_formula=f"((rho * 0 + {R_BUS}) / (1 + I * 2 * pi * f * {R_BUS} * {C_BUS_LUMPED}))",
        R_formula=f"rho * 0 + {R_BUS}",
    )
    coupling = {
        "mutual_impedance_formula": f"((rho * 0 + {R_MUT_PER_KM}) + I * 2 * pi * f * {M_PER_KM}) * l",
        "R_mutual_formula": f"({R_MUT_PER_KM}) * l",
        "M_mutual_formula": f"({M_PER_KM}) * l",
    } if mutual else {"mutual_impedance_formula": "(rho * 0 + 0.0) * l"}
    branch_type = gi.BranchType(
        name="NA2XS_300_25", grounding_conductor=True,
        self_impedance_formula=f"((rho * 0 + {R_PER_KM}) + I * 2 * pi * f * {L_PER_KM}) * l",
        R_self_formula=f"({R_PER_KM}) * l",
        L_self_formula=f"({L_PER_KM}) * l",
        C_self_formula=f"({C_PER_KM}) * l",
        **coupling,
    )
    net = gi.create_network(name="mv_ring_20", frequencies=[50.0])
    for k in range(1, N_BUSES + 1):
        gi.create_bus(name=f"b{k:02d}", type=bus_type, network=net)
    for k in range(1, N_BUSES + 1):
        nxt = (k % N_BUSES) + 1
        gi.create_branch(name=f"L_{k:02d}_{nxt:02d}", type=branch_type,
                         from_bus=f"b{k:02d}", to_bus=f"b{nxt:02d}",
                         length=SECTION_LENGTH_KM, network=net)
    if source == "current":
        gi.create_source(name="src", bus="b01", values={50.0: 1000.0 + 0.0j}, network=net)
    else:
        u_phase = 20e3 / math.sqrt(3)
        gi.create_voltage_source(
            name="vsrc", bus="b01", voltage={50.0: u_phase + 0.0j},
            source_impedance={50.0: 12.0 + 1j * 2 * math.pi * 50.0 * 3e-3}, network=net,
        )
    gi.create_fault(name="F1", bus="b10", network=net)
    return net


def stationary_epr(net, bus="b10"):
    with warnings.catch_warnings():
        # PHASE_PROXY: the ring has no phase_impedance_formula; every section is
        # the same cable, so the proxy gives the exact split (stationary notebook).
        warnings.simplefilter("ignore", gi.ModelAssumptionWarning)
        result = gi.run_fault(net, fault="F1")
    return abs(complex(next(b.uepr_freq[50.0] for b in result.buses if b.name == bus)))


def rms_of_last_period(result, bus="b10"):
    t = np.asarray(result.time_s)
    u = np.asarray(result.epr_t[bus])
    last = t >= t[-1] - 0.02 + 1e-12
    return math.sqrt(np.mean(u[last] ** 2))


net = ring(mutual=True)
net_no_mut = ring(mutual=False)

## 3. Steady state: each solver against the stationary solve

A pure 50 Hz sine of 1000 A RMS — the waveform functions take the **peak**
value, $\sqrt{2}\cdot 1000$ A — drives both solvers for ten periods. The RMS of
the EPR at `b10` over the last period is compared with the stationary solve of
the ring with and without the coupling.

In [ ]:
sine = waveforms.sinusoidal_with_dc_offset(amplitude=math.sqrt(2) * 1000.0, frequency_hz=50.0)


def steady_state(network, solver, dt):
    study = gi.TransientStudy(network, fault="F1")
    study.set_source_waveform("src", sine)
    study.set_observation(buses=["b10"])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = study.solve(t_end=0.2, dt=dt, solver=solver)
    codes = sorted({getattr(w.message, "code", "") for w in caught})
    return rms_of_last_period(result), codes


u_with, u_without = stationary_epr(net), stationary_epr(net_no_mut)
fft_rms, fft_codes = steady_state(net, "fft", 1e-4)
ss_rms, ss_codes = steady_state(net, "state_space", 2e-5)

print(f"stationary, with coupling     : {u_with:8.3f} V")
print(f"stationary, without coupling  : {u_without:8.3f} V")
print(f"FFT solver, ring with coupling: {fft_rms:8.3f} V   warnings {fft_codes}")
print(f"state space, ring with coupling: {ss_rms:7.3f} V   warnings {ss_codes}")

The FFT solver reproduces the stationary result **without** the coupling, as
its warning says; the state-space solver reproduces the stationary result
**with** it. The `PHASE_PROXY` notice of the state-space solver is the one of
the stationary notebook: the phase current divides between the two directions
of the ring, and in a ring of one cable type the proxy gives the exact split.
For a study with coupling — every cable network — use the state-space solver.

## 4. A fault current with DC offset: the effect of the coupling

An asymmetrical fault current — 1000 A peak, switched on at 20 ms with a DC
offset of 600 A decaying with 40 ms, cleared at 120 ms — drives the
state-space solver on the ring with and without the coupling.

In [ ]:
fault_current = waveforms.sinusoidal_with_dc_offset(
    amplitude=1000.0, frequency_hz=50.0, t_on=0.02, t_off=0.12,
    dc_amplitude=600.0, dc_decay_tau=0.04,
)


def state_space(network):
    study = gi.TransientStudy(network, fault="F1")
    study.set_source_waveform("src", fault_current)
    study.set_observation(buses=["b10"], branches=["L_09_10", "L_10_11"])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", gi.ModelAssumptionWarning)  # PHASE_PROXY, see above
        return study.solve(t_end=0.2, dt=5e-5, solver="state_space")


with_mut = state_space(net)
without_mut = state_space(net_no_mut)

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for result, label in ((without_mut, "without coupling"), (with_mut, "with coupling")):
    axes[0].plot(result.time_s, result.epr_t["b10"], label=label, alpha=0.85)
    axes[1].plot(result.time_s, result.i_branch_t["L_09_10"], label=label, alpha=0.85)
for ax in axes:
    ax.axvline(0.02, color="tab:red", ls=":", alpha=0.5)
    ax.axvline(0.12, color="tab:green", ls=":", alpha=0.5)
    ax.grid(True, alpha=0.3); ax.legend(loc="upper right")
axes[0].set_ylabel("EPR(b10) / V"); axes[0].set_title("Fault-bus EPR, state-space solver")
axes[1].set_ylabel("i_shield(L_09_10) / A"); axes[1].set_xlabel("t / s")
axes[1].set_title("Shield current next to the fault")
fig.tight_layout(); plt.show()

peak_with = np.max(np.abs(with_mut.epr_t["b10"]))
peak_without = np.max(np.abs(without_mut.epr_t["b10"]))
print(f"peak |EPR(b10)| with coupling {peak_with:.0f} V, without {peak_without:.0f} V")

The phase current induces a voltage along every shield, which drives most of
the fault current back through the shields instead of the soil: the coupling
lowers the peak EPR at the fault by roughly a factor of eight here, the same
order as the ratio of the two stationary results above. During the DC offset
the coupling helps less — at DC the shields carry no induced current — which
is why the EPR with coupling is lopsided at the start.

## 5. A Thévenin source switched onto the ring

A 20 kV phase voltage behind 12 Ω + 3 mH switched on at 20 ms. The
state-space solver models the coupling for current sources only; for a voltage
source it skips it and says so (`TRANSIENT_COUPLING_SKIPPED`), so this run
shows the ring **without** the shield coupling — the switching transient and
the ringing of the cable capacitance, not the coupled earthing response. Its
steady state is checked against the stationary solve of the uncoupled ring
with the same source.

In [ ]:
net_v = ring(mutual=True, source="voltage")
u_phase_peak = 20e3 / math.sqrt(3) * math.sqrt(2)
study_v = gi.TransientStudy(net_v, fault="F1")
study_v.set_source_waveform(
    "vsrc", waveforms.sinusoidal_with_dc_offset(amplitude=u_phase_peak, frequency_hz=50.0,
                                                t_on=0.02, t_off=None),
)
study_v.set_observation(buses=["b01", "b05", "b10", "b15", "b20"],
                        branches=["L_09_10", "L_10_11"])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result_v = study_v.solve(t_end=0.15, dt=2e-5, solver="state_space")
print("warnings:", sorted({getattr(w.message, "code", "") for w in caught}))

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for bus in ["b01", "b05", "b10", "b15", "b20"]:
    axes[0].plot(result_v.time_s, result_v.epr_t[bus], label=bus, alpha=0.85)
for branch in ["L_09_10", "L_10_11"]:
    axes[1].plot(result_v.time_s, result_v.i_branch_t[branch], label=branch, alpha=0.85)
for ax in axes:
    ax.axvline(0.02, color="tab:red", ls=":", alpha=0.5)
    ax.grid(True, alpha=0.3); ax.legend(loc="upper right", ncols=5, fontsize=9)
axes[0].set_ylabel("EPR / V"); axes[0].set_title("Thévenin switching: EPR across the ring (coupling skipped)")
axes[1].set_ylabel("i_shield / A"); axes[1].set_xlabel("t / s")
fig.tight_layout(); plt.show()

net_v_uncoupled = ring(mutual=False, source="voltage")
print(f"steady state RMS EPR(b10): state space {rms_of_last_period(result_v):.1f} V, "
      f"stationary without coupling {stationary_epr(net_v_uncoupled):.1f} V")

## Take-aways

* `C_self_formula` on the cable type is enough — the state-space solver lumps
  it onto the adjacent buses (pi-section); the bus impedance of the
  frequency-domain formulas carries the same capacitance.
* The state-space solver includes the phase-shield coupling for current
  sources and reproduces the stationary solve in steady state. The FFT solver
  does not include it and reproduces the stationary solve of the network
  *without* coupling; for a cable network that overestimates the EPR, here by
  a factor of three in steady state.
* The coupling lowers the EPR most for the alternating part of a fault
  current; a decaying DC offset is not reduced by it.
* For voltage sources the state-space solver does not model the coupling yet
  and warns; use a current source carrying the prospective fault current when
  the coupled response matters.